In [ ]:
# ==========================================
# Cell 1: Drive Setup & Smart Model Caching
# ==========================================
import os
from google.colab import drive

print("📁 Mounting Google Drive...")
drive.mount('/content/drive')

# 1. Project directory setup
PROJECT_DIR = '/content/drive/MyDrive/Video_Translation_Project'
MODELS_CACHE = os.path.join(PROJECT_DIR, 'Models_Cache')
STAGE_1_DIR = os.path.join(PROJECT_DIR, 'Stage_1')
STAGE_2_DIR = os.path.join(PROJECT_DIR, 'Stage_2')

for d in [MODELS_CACHE, STAGE_1_DIR, STAGE_2_DIR]:
    os.makedirs(d, exist_ok=True)

# 2. Engineering trick: Direct model downloads to Google Drive
os.environ['HF_HOME'] = MODELS_CACHE
os.environ['XDG_DATA_HOME'] = MODELS_CACHE
os.environ['COQUI_TOS_AGREED'] = "1"

print(f"✅ Permanent model caching path set to: {MODELS_CACHE}")

📁 Mounting Google Drive...
Mounted at /content/drive
✅ Permanent model caching path set to: /content/drive/MyDrive/Video_Translation_Project/Models_Cache


In [ ]:
# ==========================================
# Cell 2: Install Core Libraries (Bulletproof Version)
# ==========================================
print("[INFO] Installing system build tools...")
!sudo apt-get update -q
!sudo apt-get install -y -q espeak-ng cmake gcc g++ python3.10 python3.10-dev python3.10-distutils

print("[INFO] Setting up Python 3.10...")
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10

print("[INFO] Setting up core dependencies (Strict Numpy 1.26.4)...")
!python3.10 -m pip install -q --upgrade pip "setuptools<82" wheel Cython
# Forcing clean numpy installation to prevent ndarray errors
!python3.10 -m pip install -q --force-reinstall --no-cache-dir "numpy==1.26.4"

print("[INFO] Building and installing TTS...")
!python3.10 -m pip install -q --ignore-installed blinker TTS==0.22.0

print("[INFO] Installing pipeline models (Strict compatibility versions)...")
# Including torchcodec for audio reading, and strict tokenizers/transformers to prevent conflicts
!python3.10 -m pip install -q faster-whisper "transformers==4.44.2" "tokenizers==0.19.1" accelerate librosa soundfile torchcodec

print("\n✅ All libraries successfully installed and ready!")

[INFO] Installing system build tools...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,764 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.4 MB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http:

In [ ]:
%%writefile full_pipeline.py
# ==========================================
# Cell 3: FULL AI PIPELINE (Init + Execution)
# ==========================================
import os
import torch

# ==========================================
# PYTORCH 2.6+ COMPATIBILITY PATCH
# Bypasses the strict 'weights_only' security block for older TTS models
# ==========================================
_original_load = torch.load
def _patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_load(*args, **kwargs)
torch.load = _patched_load
# ==========================================

import librosa
import soundfile as sf
from faster_whisper import WhisperModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from TTS.api import TTS
import numpy as np

# ==========================================
# DIRECTORY PATHS & ENVIRONMENT
# ==========================================
PROJECT_DIR = '/content/drive/MyDrive/Video_Translation_Project'
MODELS_CACHE = os.path.join(PROJECT_DIR, 'Models_Cache')
STAGE_1_DIR = os.path.join(PROJECT_DIR, 'Stage_1')
STAGE_2_DIR = os.path.join(PROJECT_DIR, 'Stage_2')

os.environ['HF_HOME'] = MODELS_CACHE
os.environ['XDG_DATA_HOME'] = MODELS_CACHE
os.environ['COQUI_TOS_AGREED'] = "1"
# ==========================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Loading models into VRAM ({device})...")

# 1. Load Faster-Whisper
print("\n[1/3] Loading Faster-Whisper (Large-v3)...")
stt_model = WhisperModel("large-v3", device=device, compute_type="float16")

# 2. Load Qwen2.5-1.5B
print("\n[2/3] Loading Qwen2.5-1.5B-Instruct...")
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
llm_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

# 3. Load XTTS v2
print("\n[3/3] Loading XTTS v2...")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

print("\n🎯 All models are successfully loaded into memory and ready!")

# ==========================================
# The Golden Pipeline Execution
# ==========================================
INPUT_AUDIO = os.path.join(STAGE_1_DIR, "Clean_Vocals.wav")
TEMP_TTS_OUTPUT = os.path.join(STAGE_2_DIR, "temp_arabic.wav")
FINAL_SYNCED_OUTPUT = os.path.join(STAGE_2_DIR, "Final_Arabic_Synced.wav")

if not os.path.exists(INPUT_AUDIO):
    print(f"\n⚠️ Clean audio file not found at: {INPUT_AUDIO}")
else:
    # Step 1: Speech-to-Text
    print("\n🎙️ Step 1: Extracting English text (Faster-Whisper)...")
    segments, info = stt_model.transcribe(INPUT_AUDIO, beam_size=5, language="en")
    english_text = " ".join([segment.text for segment in segments])
    print(f"📝 Original Text: {english_text}")

    # Step 2: Translation
    print("\n🧠 Step 2: Applying strict contextual translation (Qwen2.5)...")
    system_prompt = "You are a professional English to Arabic translator. Translate the given text to natural, formal Arabic. Output ONLY the Arabic translation. Do not add introductions, explanations, quotes, or any extra words."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": english_text}
    ]
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text_input], return_tensors="pt").to(device)

    generated_ids = llm_model.generate(model_inputs.input_ids, max_new_tokens=512, temperature=0.1)
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    arabic_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    print(f"🌍 Arabic Text: {arabic_text}")

    # Step 3: Voice Cloning & TTS
    print("\n🗣️ Step 3: Cloning voice and generating speech (XTTS v2)...")
    tts.tts_to_file(text=arabic_text,
                    file_path=TEMP_TTS_OUTPUT,
                    speaker_wav=INPUT_AUDIO,
                    language="ar",
                    split_sentences=True)

    # Step 4: Time-Stretching (FIXED METHOD)
    print("\n⏱️ Step 4: Adjusting audio duration (Matching exact length)...")

    orig_y, orig_sr = librosa.load(INPUT_AUDIO, sr=None)
    orig_duration = librosa.get_duration(y=orig_y, sr=orig_sr)

    gen_y, gen_sr = librosa.load(TEMP_TTS_OUTPUT, sr=None)
    gen_duration = librosa.get_duration(y=gen_y, sr=gen_sr)

    print(f"   - English Duration : {orig_duration:.2f} seconds")
    print(f"   - Arabic Duration  : {gen_duration:.2f} seconds")

    # Calculate target length in samples
    target_samples = int(orig_duration * gen_sr)
    current_samples = len(gen_y)

    if abs(current_samples - target_samples) > (gen_sr * 0.1): # 0.1s threshold
        print("   - Applying duration adjustment...")
        # Safe method for duration adjustment that avoids numba/numpy conflicts
        stretched_y = librosa.resample(gen_y, orig_sr=gen_sr, target_sr=int(gen_sr * (current_samples / target_samples)))
        # Ensure exact length
        if len(stretched_y) > target_samples:
            stretched_y = stretched_y[:target_samples]
        elif len(stretched_y) < target_samples:
            stretched_y = np.pad(stretched_y, (0, target_samples - len(stretched_y)))
    else:
        print("   - Durations are nearly identical, no adjustment needed.")
        stretched_y = gen_y

    sf.write(FINAL_SYNCED_OUTPUT, stretched_y, gen_sr)

    print("\n🎉 Pipeline execution complete!")
    print(f"The synced Arabic file is now ready at: {FINAL_SYNCED_OUTPUT}")

Overwriting full_pipeline.py


In [ ]:
# ==========================================
# Cell 4: Run the Process!
# ==========================================
!python3.10 full_pipeline.py

/usr/local/lib/python3.10/dist-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
🚀 Loading models into VRAM (cuda)...

[1/3] Loading Faster-Whisper (Large-v3)...

[2/3] Loading Qwen2.5-1.5B-Instruct...

[3/3] Loading XTTS v2...
 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts

🎯 All models are successfully loaded into memory and ready!

🎙️ Step 1: Extracting English text (Faster-Whisper)...
📝 Original Text:  Artificial intelligence is transforming the way we live and work.

🧠 Step 2: Applying strict contextual translation (Qwen2.5)...
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe u